# SpliceAI

![SpliceAI](https://proto-bio.github.io/proto-assets/images/tool/spliceai/hero.png)

SpliceAI is a deep neural network that predicts splicing from primary DNA sequence. It serves two purposes: scoring genetic variants for their splice-altering effect via delta scores (probabilities that the variant gains or loses an acceptor/donor site), and predicting raw per-position acceptor/donor splice-site probabilities directly from a DNA sequence. It is widely used to interpret the splicing impact of non-coding and synonymous variants in clinical and research genomics.

- Paper: [Jaganathan et al., Cell 2019](https://doi.org/10.1016/j.cell.2018.12.015)
- Repository: [Illumina/SpliceAI](https://github.com/Illumina/SpliceAI)

In [1]:
from proto_tools.tools import (
    run_spliceai_score,
    run_spliceai_predict,
    SpliceAIScoreInput,
    SpliceAIScoreConfig,
    SpliceAIVariant,
    SpliceAIPredictInput,
    SpliceAIPredictConfig,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [2]:
# API reference for the variant-scoring tool
display_api_reference("spliceai-score", "input", "run_spliceai_score")
display_api_reference("spliceai-score", "config", "run_spliceai_score")
display_api_reference("spliceai-score", "output", "run_spliceai_score")

**Input** — `SpliceAIScoreInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>variants</code> | <code>list[SpliceAIVariant]</code> | required | Variants to score for splice-altering effects |

**Config** — `SpliceAIScoreConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run inference on (e.g. 'cpu', 'cuda', 'cuda:0') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>reference_fasta</code> | <code>str &#124; None</code> | <code>None</code> | Path (or AssetRef) to the reference genome FASTA; required at call time |
| <code>annotation</code> | <code>str</code> | <code>'grch38'</code> | 'grch37'/'grch38' (bundled GENCODE) or path to a custom gene annotation file |
| <code>max_distance</code> | <code>int</code> | <code>50</code> | Max distance (bp) between variant and gained/lost splice site (the -D flag) |
| <code>mask</code> | <code>bool</code> | <code>False</code> | Zero out scores for annotated-site gains and unannotated-site losses (SpliceAI -M flag) |

**Output** — `SpliceAIScoreOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[SpliceAIVariantResult]</code> | required | Per-variant SpliceAI scores (1:1 with input variants) |

**`SpliceAIVariantResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>chromosome</code> | <code>str</code> | required | Variant chromosome |
| <code>position</code> | <code>int</code> | required | Variant position (1-based) |
| <code>ref</code> | <code>str</code> | required | Reference allele |
| <code>alt</code> | <code>str</code> | required | Alternate allele |
| <code>scores</code> | <code>list[SpliceAIGeneScore]</code> | required | Per-gene SpliceAI delta scores (empty if no gene overlap) |
| <code>metrics</code> | <code>SpliceAIScoreMetrics</code> | required | Per-variant scalar metric (max delta score) |

**`SpliceAIGeneScore`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>allele</code> | <code>str</code> | required | Alternate allele these scores correspond to |
| <code>symbol</code> | <code>str</code> | required | Gene symbol the variant was scored against |
| <code>ds_ag</code> | <code>float &#124; None</code> | required | Delta score for acceptor gain (0-1) |
| <code>ds_al</code> | <code>float &#124; None</code> | required | Delta score for acceptor loss (0-1) |
| <code>ds_dg</code> | <code>float &#124; None</code> | required | Delta score for donor gain (0-1) |
| <code>ds_dl</code> | <code>float &#124; None</code> | required | Delta score for donor loss (0-1) |
| <code>dp_ag</code> | <code>int &#124; None</code> | required | Delta position for acceptor gain (bp from variant) |
| <code>dp_al</code> | <code>int &#124; None</code> | required | Delta position for acceptor loss (bp from variant) |
| <code>dp_dg</code> | <code>int &#124; None</code> | required | Delta position for donor gain (bp from variant) |
| <code>dp_dl</code> | <code>int &#124; None</code> | required | Delta position for donor loss (bp from variant) |

**Metrics** — `SpliceAIScoreMetrics` (one set per `results` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>max_delta_score</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> |  | Maximum delta score (acceptor/donor gain/loss) across overlapping genes |

In [3]:
# API reference for the raw splice-site prediction tool
display_api_reference("spliceai-predict", "input", "run_spliceai_predict")
display_api_reference("spliceai-predict", "config", "run_spliceai_predict")
display_api_reference("spliceai-predict", "output", "run_spliceai_predict")

**Input** — `SpliceAIPredictInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA sequence(s) to predict splice-site probabilities for |

**Config** — `SpliceAIPredictConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run inference on (e.g. 'cpu', 'cuda', 'cuda:0') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |

**Output** — `SpliceAIPredictOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>predictions</code> | <code>list[list[list[float]]]</code> | required | Per-sequence, per-position [neither, acceptor, donor] probabilities |

## Variant scoring

Variant scoring needs a reference genome FASTA and a gene annotation: SpliceAI extracts the wild-type sequence window around each variant from the genome, applies the alternate allele, and compares the ensemble's predictions to derive delta scores for every overlapping gene. Here we bundle the real **HBB locus** (GRCh38 chr11, in transcription orientation) with a matching HBB annotation, and score **IVS-I-1** (`HBB:c.92+1G>A`) — the classic β-thalassemia variant that abolishes the intron-1 donor site. Real analyses use the full human genome (`grch37`/`grch38`) and the bundled GENCODE annotation.

In [4]:
config = SpliceAIScoreConfig(
    reference_fasta="hbb_locus.fasta",
    annotation="hbb_annotation.txt",
    device="cpu",  # CPU so this runs on hosts without a GPU
)
# IVS-I-1 (HBB:c.92+1G>A): the classic beta-thalassemia variant that abolishes the
# HBB intron-1 donor splice site.
inputs = SpliceAIScoreInput(
    variants=[SpliceAIVariant(chromosome="HBB", position=5143, ref="G", alt="A")]
)

result = run_spliceai_score(inputs, config)

variant_result = result.results[0]
print(f"Variant: {variant_result.chromosome}:{variant_result.position} {variant_result.ref}>{variant_result.alt}")
print(f"Max delta score: {variant_result.metrics['max_delta_score']}")
print("\nPer-gene delta scores:")
for gene in variant_result.scores:
    print(
        f"  {gene.symbol} (allele {gene.allele}): "
        f"AG={gene.ds_ag:.3f} AL={gene.ds_al:.3f} "
        f"DG={gene.ds_dg:.3f} DL={gene.ds_dl:.3f}"
    )

Running run_spliceai_score [00:00]

Variant: HBB:5143 G>A
Max delta score: 0.98

Per-gene delta scores:
  HBB (allele A): AG=0.000 AL=0.000 DG=0.480 DL=0.980


## Raw splice-site prediction

Raw prediction runs directly on a DNA sequence and needs no reference genome. SpliceAI pads 5000 bp of context on each side internally, so the output covers every position of the input sequence with a `[neither, acceptor, donor]` probability triple.

In [5]:
from pathlib import Path
import numpy as np

# Real HBB locus (GRCh38 chr11); SpliceAI flags the HBB exon/intron boundaries.
sequence = "".join(l for l in Path("hbb_locus.fasta").read_text().splitlines() if not l.startswith(">"))
predict_result = run_spliceai_predict(
    SpliceAIPredictInput(sequences=[sequence]),
    SpliceAIPredictConfig(device="cpu"),
)

probs = np.array(predict_result.predictions[0])  # channels: [neither, acceptor, donor]
print(f"Prediction shape: {probs.shape}  # [len, 3] -> [neither, acceptor, donor]")
print(f"Sequence length: {len(sequence)}")
print(f"Max acceptor probability (channel 1): {probs[:, 1].max():.4f} at position {probs[:, 1].argmax()}")
print(f"Max donor probability (channel 2): {probs[:, 2].max():.4f} at position {probs[:, 2].argmax()}")

Running run_spliceai_predict [00:00]

Prediction shape: (12071, 3)  # [len, 3] -> [neither, acceptor, donor]
Sequence length: 12071
Max acceptor probability (channel 1): 0.9957 at position 5272
Max donor probability (channel 2): 0.9995 at position 5494


In [6]:
# Export the variant scores. Supported formats are listed by the output object.
import tempfile
from pathlib import Path

print(f"Supported formats: {result.output_format_options} (default: {result.output_format_default})")

with tempfile.TemporaryDirectory() as tmp_dir:
    result.export("spliceai_scores", export_path=tmp_dir, file_format="vcf")
    exported = Path(tmp_dir) / "spliceai_scores.vcf"
    print(f"Exported to {exported}")
    print(exported.read_text())

Supported formats: ['json', 'csv', 'vcf'] (default: json)
Exported to /tmp/tmp9f8muvdd/spliceai_scores.vcf
##fileformat=VCFv4.2
##INFO=<ID=SpliceAI,Number=.,Type=String,Description="SpliceAI variant annotation. These include delta scores (DS) and delta positions (DP) for acceptor gain (AG), acceptor loss (AL), donor gain (DG), and donor loss (DL). Format: ALLELE|SYMBOL|DS_AG|DS_AL|DS_DG|DS_DL|DP_AG|DP_AL|DP_DG|DP_DL">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
HBB	5143	.	G	A	.	.	SpliceAI=A|HBB|0.00|0.00|0.48|0.98|-39|-1|-17|-1

